In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q opencv-python mediapipe scikit-learn matplotlib seaborn

In [3]:
import cv2
import mediapipe as mp
import torch
import sklearn
import matplotlib

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("PyTorch:", torch.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Everything imported successfully ✅")

OpenCV: 5.0.0
MediaPipe: 1.0.1
PyTorch: 2.11.0+cu128
Scikit-learn: 1.6.1
Everything imported successfully ✅


In [5]:
from datasets import load_dataset

dataset = load_dataset("ai4bharat/INCLUDE")

print(dataset)

README.md:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 78.2kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/val-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 13.1kB            

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3816 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/425 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1009 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['parent_label', 'label', 'video_path', 'include_50'],
        num_rows: 3816
    })
    val: Dataset({
        features: ['parent_label', 'label', 'video_path', 'include_50'],
        num_rows: 425
    })
    test: Dataset({
        features: ['parent_label', 'label', 'video_path', 'include_50'],
        num_rows: 1009
    })
})


In [6]:
labels = set()

for split in dataset:
    labels.update(dataset[split]["label"])

print("Total unique labels:", len(labels))

for label in sorted(labels):
    print(label)

Total unique labels: 263
1. Dog
1. Religion
1. loud
10. Energy
10. Mean
10. Plane
11. Car
11. War
11. rich
12. Peace
12. Truck
12. poor
13. Attack
13. Bicycle
13. thick
14. Bus
14. Election
14. thin
15. Boat
15. Newspaper
15. expensive
16. Gun
16. cheap
16. train ticket
17. Sport
17. Transportation
17. flat
18. City
18. Exercise
18. curved
19. Ball
19. House
19. male
2. Cat
2. Death
2. quiet
20. Price
20. Street or Road
20. female
21. Sign
21. Train Station
21. tight
22. Restaurant
22. Science
22. loose
23. Court
23. God
23. high
24. School
24. Table
24. low
25. Chair
25. Office
25. soft
26. Bed
26. University
26. hard
27. Dream
27. Park
27. deep
28. Store or Shop
28. Window
28. shallow
29. Door
29. Library
29. clean
3. Fish
3. Medicine
3. happy
30. Bedroom
30. Hospital
30. dirty
31. Kitchen
31. Temple
31. strong
32. Bathroom
32. Market
32. weak
33. India
33. Pencil
33. dead
34. Ground
34. Pen
34. alive
35. Bank
35. Photograph
35. heavy
36. Location
36. Soap
36. light
37. Book
37. Hat


In [7]:
# Find the actual labels corresponding to our target words

target_words = [
    "Doctor",
    "Patient",
    "Hospital",
    "Medicine",
    "sick",
    "healthy",
    "Today",
    "Tomorrow"
]

all_labels = set()

for split in dataset:
    all_labels.update(dataset[split]["label"])

for word in target_words:
    matches = [
        label for label in all_labels
        if label.split(". ", 1)[-1].strip().lower() == word.lower()
    ]

    print(f"{word:10} → {matches}")

Doctor     → ['87. Doctor']
Patient    → ['88. Patient']
Hospital   → ['30. Hospital']
Medicine   → ['3. Medicine']
sick       → ['98. sick']
healthy    → ['99. healthy']
Today      → ['73. Today']
Tomorrow   → ['74. Tomorrow']


In [8]:
# Get the exact dataset labels
selected_labels = []

for word in target_words:
    for label in all_labels:
        clean_label = label.split(". ", 1)[-1].strip()

        if clean_label.lower() == word.lower():
            selected_labels.append(label)

print("Selected labels:")
for label in sorted(selected_labels):
    print(label)

Selected labels:
3. Medicine
30. Hospital
73. Today
74. Tomorrow
87. Doctor
88. Patient
98. sick
99. healthy


In [9]:
selected_data = {}

for split in dataset:
    selected_data[split] = dataset[split].filter(
        lambda example: example["label"] in selected_labels
    )

print(selected_data)

Filter:   0%|          | 0/3816 [00:00<?, ? examples/s]

Filter:   0%|          | 0/425 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1009 [00:00<?, ? examples/s]

{'train': Dataset({
    features: ['parent_label', 'label', 'video_path', 'include_50'],
    num_rows: 99
}), 'val': Dataset({
    features: ['parent_label', 'label', 'video_path', 'include_50'],
    num_rows: 10
}), 'test': Dataset({
    features: ['parent_label', 'label', 'video_path', 'include_50'],
    num_rows: 23
})}


In [10]:
for split in selected_data:
    print(f"\n{split.upper()}")

    counts = {}

    for label in selected_data[split]["label"]:
        counts[label] = counts.get(label, 0) + 1

    for label, count in sorted(counts.items()):
        print(f"{label:20} : {count}")


TRAIN
3. Medicine          : 10
30. Hospital         : 15
73. Today            : 12
74. Tomorrow         : 12
87. Doctor           : 10
88. Patient          : 10
98. sick             : 15
99. healthy          : 15

VAL
3. Medicine          : 1
30. Hospital         : 1
73. Today            : 1
74. Tomorrow         : 1
87. Doctor           : 1
88. Patient          : 1
98. sick             : 2
99. healthy          : 2

TEST
3. Medicine          : 3
30. Hospital         : 4
73. Today            : 1
74. Tomorrow         : 1
87. Doctor           : 3
88. Patient          : 3
98. sick             : 4
99. healthy          : 4


In [11]:
print(dataset["train"].column_names)

['parent_label', 'label', 'video_path', 'include_50']


In [12]:
for split in selected_data:
    print(f"\n===== {split.upper()} =====")

    for i in range(min(10, len(selected_data[split]))):
        example = selected_data[split][i]

        print(
            f"{example['label']:20} → {example['video_path']}"
        )


===== TRAIN =====
74. Tomorrow         → Days_and_Time/74. Tomorrow/MVI_4619.MOV
73. Today            → Days_and_Time/73. Today/MVI_5471.MOV
98. sick             → Adjectives/98. sick/MVI_5253.MOV
98. sick             → Adjectives/98. sick/MVI_5171.MOV
74. Tomorrow         → Days_and_Time/74. Tomorrow/MVI_5035.MOV
99. healthy          → Adjectives/99. healthy/MVI_9286.MOV
87. Doctor           → Jobs/87. Doctor/MVI_5325.MOV
98. sick             → Adjectives/98. sick/MVI_9362.MOV
88. Patient          → Jobs/88. Patient/MVI_4760.MOV
98. sick             → Adjectives/98. sick/MVI_9442.MOV

===== VAL =====
98. sick             → Adjectives/98. sick/MVI_9444.MOV
73. Today            → Days_and_Time/73. Today/MVI_5031.MOV
3. Medicine          → Society/3. Medicine/MVI_8665.MP4
99. healthy          → Adjectives/99. healthy/MVI_5333.MOV
98. sick             → Adjectives/98. sick/MVI_5329.MOV
88. Patient          → Jobs/88. Patient/MVI_4475.MOV
30. Hospital         → Places/30. Hospital/MVI_356

In [13]:
print(selected_data["train"][0])

{'parent_label': 'Days_and_Time', 'label': '74. Tomorrow', 'video_path': 'Days_and_Time/74. Tomorrow/MVI_4619.MOV', 'include_50': False}


In [15]:
import requests

zenodo_api = "https://zenodo.org/api/records/4010759"

response = requests.get(zenodo_api)
response.raise_for_status()

zenodo = response.json()

print("Available files on Zenodo:\n")

for f in zenodo["files"]:
    print(f["key"])

Available files on Zenodo:

Adjectives_3of8.zip
Adjectives_4of8.zip
Adjectives_8of8.zip
Home_4of4.zip
Adjectives_5of8.zip
Adjectives_6of8.zip
Adjectives_7of8.zip
Pronouns_2of2.zip
Pronouns_1of2.zip
Society_2of3.zip
Places_4of4.zip
Places_3of4.zip
Society_1of3.zip
Seasons_1of1.zip
Places_2of4.zip
README.md
Places_1of4.zip
Society_3of3.zip
Days_and_Time_3of3.zip
People_5of5.zip
People_4of5.zip
People_3of5.zip
Days_and_Time_2of3.zip
Days_and_Time_1of3.zip
People_2of5.zip
Colours_2of2.zip
People_1of5.zip
Electronics_1of2.zip
Means_of_Transportation_2of2.zip
Means_of_Transportation_1of2.zip
Colours_1of2.zip
Electronics_2of2.zip
Animals_1of2.zip
Greetings_1of2.zip
Clothes_2of2.zip
Jobs_2of2.zip
Jobs_1of2.zip
Clothes_1of2.zip
Animals_2of2.zip
Greetings_2of2.zip
Home_1of4.zip
Home_2of4.zip
Home_3of4.zip
Adjectives_1of8.zip
Adjectives_2of8.zip
download_data.sh


In [3]:
import requests
import os
from tqdm.auto import tqdm

# Zenodo INCLUDE dataset
ZENODO_API = "https://zenodo.org/api/records/4010759"

filename = "Jobs_1of2.zip"
download_dir = "/content/include_zips"
os.makedirs(download_dir, exist_ok=True)

# Get Zenodo metadata
response = requests.get(ZENODO_API)
response.raise_for_status()
zenodo = response.json()

# Find our ZIP
file_info = next(
    f for f in zenodo["files"]
    if f["key"] == filename
)

url = file_info["links"]["self"]
output_path = os.path.join(download_dir, filename)

print(f"Downloading: {filename}")
print(f"Saving to: {output_path}\n")

# Download with progress bar
with requests.get(url, stream=True) as r:
    r.raise_for_status()

    total_size = int(r.headers.get("content-length", 0))

    with open(output_path, "wb") as f:
        with tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=filename
        ) as progress:

            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    progress.update(len(chunk))

print("\n✅ Download complete!")

size_mb = os.path.getsize(output_path) / (1024 ** 2)
print(f"File size: {size_mb:.2f} MB")

Downloading: Jobs_1of2.zip
Saving to: /content/include_zips/Jobs_1of2.zip



Jobs_1of2.zip:   0%|          | 0.00/1.40G [00:00<?, ?B/s]


✅ Download complete!
File size: 1433.48 MB


In [14]:
import os

path = "/content/include_zips/Jobs_1of2.zip"

print("Exists:", os.path.exists(path))
print("Size:", os.path.getsize(path) / (1024**2), "MB")

Exists: True
Size: 1433.4844770431519 MB


In [15]:
import zipfile

zip_path = "/content/include_zips/Jobs_1of2.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

print("Total files:", len(files))
print("\nFirst 30 files:")
for f in files[:30]:
    print(f)


Total files: 122

First 30 files:
Jobs/
Jobs/84. Teacher/
Jobs/84. Teacher/MVI_4457.MOV
Jobs/84. Teacher/MVI_4458.MOV
Jobs/84. Teacher/MVI_4459.MOV
Jobs/84. Teacher/MVI_4460.MOV
Jobs/84. Teacher/MVI_4744.MOV
Jobs/84. Teacher/MVI_4745.MOV
Jobs/84. Teacher/MVI_4746.MOV
Jobs/84. Teacher/MVI_5310.MOV
Jobs/84. Teacher/MVI_5311.MOV
Jobs/84. Teacher/MVI_5312.MOV
Jobs/84. Teacher/MVI_5313.MOV
Jobs/84. Teacher/MVI_8864.MP4
Jobs/84. Teacher/MVI_8865.MP4
Jobs/84. Teacher/MVI_8866.MP4
Jobs/85. Student/
Jobs/85. Student/MVI_4462.MOV
Jobs/85. Student/MVI_4463.MOV
Jobs/85. Student/MVI_4464.MOV
Jobs/85. Student/MVI_4465.MOV
Jobs/85. Student/MVI_4747.MOV
Jobs/85. Student/MVI_4748.MOV
Jobs/85. Student/MVI_4749.MOV
Jobs/85. Student/MVI_5314.MOV
Jobs/85. Student/MVI_5315.MOV
Jobs/85. Student/MVI_5316.MOV
Jobs/85. Student/MVI_5317.MOV
Jobs/85. Student/MVI_8868.MP4
Jobs/85. Student/MVI_8869.MP4


In [16]:
print("\n===== DOCTOR =====")

doctor_files = [
    f for f in files
    if "Doctor" in f
]

for f in doctor_files[:20]:
    print(f)

print("\nTotal Doctor videos:", len(doctor_files))


print("\n===== PATIENT =====")

patient_files = [
    f for f in files
    if "Patient" in f
]

for f in patient_files[:20]:
    print(f)

print("\nTotal Patient videos:", len(patient_files))


===== DOCTOR =====
Jobs/87. Doctor/
Jobs/87. Doctor/MVI_4470.MOV
Jobs/87. Doctor/MVI_4471.MOV
Jobs/87. Doctor/MVI_4472.MOV
Jobs/87. Doctor/MVI_4473.MOV
Jobs/87. Doctor/MVI_4754.MOV
Jobs/87. Doctor/MVI_4755.MOV
Jobs/87. Doctor/MVI_4757.MOV
Jobs/87. Doctor/MVI_5322.MOV
Jobs/87. Doctor/MVI_5323.MOV
Jobs/87. Doctor/MVI_5324.MOV
Jobs/87. Doctor/MVI_5325.MOV
Jobs/87. Doctor/MVI_8874.MP4
Jobs/87. Doctor/MVI_8875.MP4
Jobs/87. Doctor/MVI_8876.MP4

Total Doctor videos: 15

===== PATIENT =====
Jobs/88. Patient/
Jobs/88. Patient/MVI_4474.MOV
Jobs/88. Patient/MVI_4475.MOV
Jobs/88. Patient/MVI_4476.MOV
Jobs/88. Patient/MVI_4477.MOV
Jobs/88. Patient/MVI_4758.MOV
Jobs/88. Patient/MVI_4759.MOV
Jobs/88. Patient/MVI_4760.MOV
Jobs/88. Patient/MVI_5326.MOV
Jobs/88. Patient/MVI_5327.MOV
Jobs/88. Patient/MVI_5328.MOV
Jobs/88. Patient/MVI_5329.MOV
Jobs/88. Patient/MVI_8878.MP4
Jobs/88. Patient/MVI_8879.MP4
Jobs/88. Patient/MVI_8880.MP4

Total Patient videos: 15


In [17]:
import zipfile
import os

zip_path = "/content/include_zips/Jobs_1of2.zip"
extract_dir = "/content/include_selected"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    selected_files = [
        f for f in z.namelist()
        if ("87. Doctor/" in f or "88. Patient/" in f)
        and f.lower().endswith((".mov", ".mp4"))
    ]

    print("Files to extract:", len(selected_files))

    for f in selected_files:
        z.extract(f, extract_dir)

print("\n✅ Extraction complete!")

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith((".mov", ".mp4")):
            print(os.path.join(root, file))

Files to extract: 28

✅ Extraction complete!
/content/include_selected/Jobs/87. Doctor/MVI_5322.MOV
/content/include_selected/Jobs/87. Doctor/MVI_8874.MP4
/content/include_selected/Jobs/87. Doctor/MVI_4473.MOV
/content/include_selected/Jobs/87. Doctor/MVI_5324.MOV
/content/include_selected/Jobs/87. Doctor/MVI_4754.MOV
/content/include_selected/Jobs/87. Doctor/MVI_4470.MOV
/content/include_selected/Jobs/87. Doctor/MVI_5323.MOV
/content/include_selected/Jobs/87. Doctor/MVI_8876.MP4
/content/include_selected/Jobs/87. Doctor/MVI_5325.MOV
/content/include_selected/Jobs/87. Doctor/MVI_4755.MOV
/content/include_selected/Jobs/87. Doctor/MVI_8875.MP4
/content/include_selected/Jobs/87. Doctor/MVI_4757.MOV
/content/include_selected/Jobs/87. Doctor/MVI_4471.MOV
/content/include_selected/Jobs/87. Doctor/MVI_4472.MOV
/content/include_selected/Jobs/88. Patient/MVI_4475.MOV
/content/include_selected/Jobs/88. Patient/MVI_4477.MOV
/content/include_selected/Jobs/88. Patient/MVI_4758.MOV
/content/include_

In [18]:
import cv2
import os

video_path = "/content/include_selected/Jobs/87. Doctor/MVI_5322.MOV"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("❌ Could not open video")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0

    print("✅ Video opened successfully!")
    print(f"FPS: {fps}")
    print(f"Total frames: {frame_count}")
    print(f"Resolution: {width} × {height}")
    print(f"Duration: {duration:.2f} seconds")

cap.release()

✅ Video opened successfully!
FPS: 25.0
Total frames: 50
Resolution: 1920 × 1080
Duration: 2.00 seconds


In [19]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt

video_path = "/content/include_selected/Jobs/87. Doctor/MVI_5322.MOV"

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5
)

# Read the middle frame
cap = cv2.VideoCapture(video_path)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
middle_frame = total_frames // 2

cap.set(cv2.CAP_PROP_POS_FRAMES, middle_frame)
success, frame = cap.read()
cap.release()

if not success:
    print("❌ Could not read frame")
else:
    # OpenCV uses BGR → MediaPipe expects RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        print(f"✅ Detected {len(results.multi_hand_landmarks)} hand(s)")

        for hand_idx, hand_landmarks in enumerate(results.multi_hand_landmarks):

            print(f"\nHand {hand_idx + 1}:")

            for landmark_idx, landmark in enumerate(hand_landmarks.landmark[:5]):
                print(
                    f"Landmark {landmark_idx}: "
                    f"x={landmark.x:.4f}, "
                    f"y={landmark.y:.4f}, "
                    f"z={landmark.z:.4f}"
                )

    else:
        print("⚠️ No hands detected in the middle frame")

    # Display the frame
    plt.figure(figsize=(10, 6))
    plt.imshow(rgb_frame)
    plt.axis("off")
    plt.title("Doctor - Middle Frame")
    plt.show()

hands.close()

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [20]:
!pip install mediapipe

In [21]:
import os
import urllib.request

model_path = "/content/hand_landmarker.task"

url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"

if not os.path.exists(model_path):
    print("Downloading Hand Landmarker model...")
    urllib.request.urlretrieve(url, model_path)
    print("✅ Model downloaded!")
else:
    print("✅ Model already exists!")

print("Model size:", os.path.getsize(model_path) / (1024**2), "MB")

✅ Model downloaded!
Model size: 7.456879615783691 MB


In [22]:
import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("Tasks available:", hasattr(mp, "tasks"))

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("✅ MediaPipe Tasks API working!")

MediaPipe version: 1.0.1
Tasks available: True
✅ MediaPipe Tasks API working!


In [23]:
import cv2
import mediapipe as mp
import numpy as np

# ============================================
# INPUT VIDEO
# ============================================

video_path = "/content/include_selected/Jobs/87. Doctor/MVI_5322.MOV"

# ============================================
# MEDIAPIPE HAND LANDMARKER
# ============================================

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="/content/hand_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# ============================================
# PROCESS VIDEO
# ============================================

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video FPS: {fps}")
print(f"Total frames: {total_frames}")

landmark_sequences = []
frames_processed = 0
frames_with_hands = 0

with HandLandmarker.create_from_options(options) as landmarker:

    while True:
        success, frame = cap.read()

        if not success:
            break

        # BGR → RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        # Timestamp must increase for VIDEO mode
        timestamp_ms = int((frames_processed / fps) * 1000)

        result = landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        # ----------------------------------------
        # Create fixed 2-hand representation
        # ----------------------------------------

        frame_landmarks = np.zeros((2, 21, 3), dtype=np.float32)

        if result.hand_landmarks:

            frames_with_hands += 1

            for hand_idx, hand in enumerate(result.hand_landmarks[:2]):

                for landmark_idx, landmark in enumerate(hand):
                    frame_landmarks[hand_idx, landmark_idx] = [
                        landmark.x,
                        landmark.y,
                        landmark.z
                    ]

        landmark_sequences.append(frame_landmarks)

        frames_processed += 1

cap.release()

landmark_sequences = np.array(landmark_sequences)

print("\n==============================")
print("PROCESSING COMPLETE")
print("==============================")
print("Frames processed:", frames_processed)
print("Frames with hands:", frames_with_hands)
print("Frames without hands:", frames_processed - frames_with_hands)
print("Output shape:", landmark_sequences.shape)
print("Expected shape:", (total_frames, 2, 21, 3))

Video FPS: 25.0
Total frames: 50

PROCESSING COMPLETE
Frames processed: 50
Frames with hands: 33
Frames without hands: 17
Output shape: (50, 2, 21, 3)
Expected shape: (50, 2, 21, 3)


In [24]:
import cv2
import mediapipe as mp
import numpy as np
import os
from tqdm.auto import tqdm

# ============================================
# SETTINGS
# ============================================

DATASET_DIR = "/content/include_selected"

VIDEO_EXTENSIONS = (".mov", ".mp4")

LABEL_MAP = {
    "87. Doctor": 0,
    "88. Patient": 1
}

SEQUENCE_LENGTH = 50

# ============================================
# MEDIAPIPE SETUP
# ============================================

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="/content/hand_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# ============================================
# FIND ALL VIDEOS
# ============================================

video_files = []

for root, dirs, files in os.walk(DATASET_DIR):

    for file in files:

        if file.lower().endswith(VIDEO_EXTENSIONS):

            full_path = os.path.join(root, file)

            if "87. Doctor" in full_path:
                label = 0

            elif "88. Patient" in full_path:
                label = 1

            else:
                continue

            video_files.append((full_path, label))


print("Total videos:", len(video_files))
print("Doctor:", sum(label == 0 for _, label in video_files))
print("Patient:", sum(label == 1 for _, label in video_files))


# ============================================
# FUNCTION: PROCESS ONE VIDEO
# ============================================

def extract_landmarks(video_path):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 25

    frames = []

    while True:

        success, frame = cap.read()

        if not success:
            break

        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        frames.append(mp_image)

    cap.release()

    # ----------------------------------------
    # Sample exactly SEQUENCE_LENGTH frames
    # ----------------------------------------

    if len(frames) == 0:
        return None

    indices = np.linspace(
        0,
        len(frames) - 1,
        SEQUENCE_LENGTH
    ).astype(int)

    selected_frames = [
        frames[i]
        for i in indices
    ]

    sequence = []

    with HandLandmarker.create_from_options(options) as landmarker:

        for frame_idx, mp_image in enumerate(selected_frames):

            timestamp_ms = frame_idx * 40  # ~25 FPS

            result = landmarker.detect_for_video(
                mp_image,
                timestamp_ms
            )

            frame_landmarks = np.zeros(
                (2, 21, 3),
                dtype=np.float32
            )

            if result.hand_landmarks:

                for hand_idx, hand in enumerate(
                    result.hand_landmarks[:2]
                ):

                    for landmark_idx, landmark in enumerate(hand):

                        frame_landmarks[
                            hand_idx,
                            landmark_idx
                        ] = [
                            landmark.x,
                            landmark.y,
                            landmark.z
                        ]

            sequence.append(frame_landmarks)

    return np.array(sequence)


# ============================================
# PROCESS ALL VIDEOS
# ============================================

X = []
y = []
paths = []

for video_path, label in tqdm(
    video_files,
    desc="Processing videos"
):

    try:

        landmarks = extract_landmarks(video_path)

        if landmarks is not None:

            X.append(landmarks)
            y.append(label)
            paths.append(video_path)

    except Exception as e:

        print("\n❌ Error:", video_path)
        print(e)


# ============================================
# CONVERT TO NUMPY
# ============================================

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)


print("\n==============================")
print("DATASET READY")
print("==============================")

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass distribution:")

print("Doctor :", np.sum(y == 0))
print("Patient:", np.sum(y == 1))

Total videos: 28
Doctor: 14
Patient: 14


Processing videos:   0%|          | 0/28 [00:00<?, ?it/s]


DATASET READY
X shape: (28, 50, 2, 21, 3)
y shape: (28,)

Class distribution:
Doctor : 14
Patient: 14


In [25]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --------------------------------------------
# Flatten each frame's hand landmarks
# 2 hands × 21 landmarks × 3 coordinates
# = 126 features per frame
# --------------------------------------------

X_flat = X.reshape(X.shape[0], X.shape[1], -1)

print("Original X:", X.shape)
print("LSTM input:", X_flat.shape)

# --------------------------------------------
# Train / validation / test split
# --------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X_flat,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\nSplit:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nClass distribution:")
print("Train - Doctor:", np.sum(y_train == 0))
print("Train - Patient:", np.sum(y_train == 1))
print("Val   - Doctor:", np.sum(y_val == 0))
print("Val   - Patient:", np.sum(y_val == 1))
print("Test  - Doctor:", np.sum(y_test == 0))
print("Test  - Patient:", np.sum(y_test == 1))

Original X: (28, 50, 2, 21, 3)
LSTM input: (28, 50, 126)

Split:
Train: (19, 50, 126) (19,)
Validation: (4, 50, 126) (4,)
Test: (5, 50, 126) (5,)

Class distribution:
Train - Doctor: 9
Train - Patient: 10
Val   - Doctor: 2
Val   - Patient: 2
Test  - Doctor: 3
Test  - Patient: 2


In [26]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report

# ============================================
# 1. CHECK GPU
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU not available")


# ============================================
# 2. CONVERT NUMPY → PYTORCH
# ============================================

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


# ============================================
# 3. DATA LOADERS
# ============================================

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)


# ============================================
# 4. LSTM MODEL
# ============================================

class ISLLSTM(nn.Module):

    def __init__(
        self,
        input_size=126,
        hidden_size=64,
        num_layers=2,
        num_classes=2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, x):

        # x shape:
        # batch × sequence × features

        output, (hidden, cell) = self.lstm(x)

        # Last time step
        x = output[:, -1, :]

        x = self.dropout(x)

        x = self.fc(x)

        return x


model = ISLLSTM().to(device)

print("\nModel:")
print(model)


# ============================================
# 5. LOSS + OPTIMIZER
# ============================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ============================================
# 6. TRAINING
# ============================================

EPOCHS = 30

for epoch in range(EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------

    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(
            outputs,
            batch_y
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        train_correct += (
            predictions == batch_y
        ).sum().item()

        train_total += batch_y.size(0)

    train_accuracy = (
        train_correct / train_total
    )


    # -------------------------
    # VALIDATION
    # -------------------------

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_X, batch_y in val_loader:

            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_X)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_correct += (
                predictions == batch_y
            ).sum().item()

            val_total += batch_y.size(0)

    val_accuracy = (
        val_correct / val_total
    )


    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"| Loss: {train_loss / len(train_loader):.4f} "
        f"| Train Acc: {train_accuracy:.2%} "
        f"| Val Acc: {val_accuracy:.2%}"
    )

Device: cuda
GPU: Tesla T4

Model:
ISLLSTM(
  (lstm): LSTM(126, 64, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)
Epoch 01/30 | Loss: 0.6987 | Train Acc: 42.11% | Val Acc: 50.00%
Epoch 02/30 | Loss: 0.7003 | Train Acc: 42.11% | Val Acc: 50.00%
Epoch 03/30 | Loss: 0.6827 | Train Acc: 57.89% | Val Acc: 50.00%
Epoch 04/30 | Loss: 0.6787 | Train Acc: 57.89% | Val Acc: 50.00%
Epoch 05/30 | Loss: 0.6849 | Train Acc: 47.37% | Val Acc: 50.00%
Epoch 06/30 | Loss: 0.6545 | Train Acc: 73.68% | Val Acc: 75.00%
Epoch 07/30 | Loss: 0.6314 | Train Acc: 73.68% | Val Acc: 75.00%
Epoch 08/30 | Loss: 0.5381 | Train Acc: 78.95% | Val Acc: 100.00%
Epoch 09/30 | Loss: 0.6243 | Train Acc: 73.68% | Val Acc: 75.00%
Epoch 10/30 | Loss: 0.5971 | Train Acc: 68.42% | Val Acc: 50.00%
Epoch 11/30 | Loss: 0.6723 | Train Acc: 63.16% | Val Acc: 50.00%
Epoch 12/30 | Loss: 0.6644 | Train Acc: 63.16% | Val Acc: 50.00%
Epo

In [27]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ============================================
# TEST MODEL
# ============================================

model.eval()

all_predictions = []
all_actual = []
all_probabilities = []

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(device)

        outputs = model(batch_X)

        probabilities = torch.softmax(outputs, dim=1)

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_actual.extend(
            batch_y.numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


# ============================================
# RESULTS
# ============================================

test_accuracy = accuracy_score(
    all_actual,
    all_predictions
)

print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(f"Test Accuracy: {test_accuracy:.2%}")

print("\nClassification Report:")

print(
    classification_report(
        all_actual,
        all_predictions,
        target_names=["Doctor", "Patient"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        all_actual,
        all_predictions
    )
)


TEST RESULTS
Test Accuracy: 20.00%

Classification Report:
              precision    recall  f1-score   support

      Doctor       0.00      0.00      0.00         3
     Patient       0.25      0.50      0.33         2

    accuracy                           0.20         5
   macro avg       0.12      0.25      0.17         5
weighted avg       0.10      0.20      0.13         5


Confusion Matrix:
[[0 3]
 [1 1]]


In [ ]:
import requests
import os
from tqdm.auto import tqdm

zenodo_api = "https://zenodo.org/api/records/4010759"

response = requests.get(zenodo_api)
response.raise_for_status()

zenodo = response.json()

download_dir = "/content/include_zips"
os.makedirs(download_dir, exist_ok=True)

# Download category archives
required_archives = [
    "Society_3of3.zip",

    "Places_4of4.zip",

    "Days_and_Time_3of3.zip",

    "Adjectives_8of8.zip"
]

zenodo_files = {
    f["key"]: f["links"]["self"]
    for f in zenodo["files"]
}

for filename in required_archives:

    if filename not in zenodo_files:
        print("❌ Not found:", filename)
        continue

    output_path = os.path.join(
        download_dir,
        filename
    )

    if os.path.exists(output_path):

        size_mb = os.path.getsize(output_path) / (1024**2)

        print(
            f"✅ Already exists: "
            f"{filename} ({size_mb:.1f} MB)"
        )

        continue

    url = zenodo_files[filename]

    print(f"\n⬇️ Downloading {filename}")

    with requests.get(
        url,
        stream=True
    ) as r:

        r.raise_for_status()

        total = int(
            r.headers.get(
                "content-length",
                0
            )
        )

        downloaded = 0

        with open(
            output_path,
            "wb"
        ) as f:

            with tqdm(
                total=total,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc=filename
            ) as pbar:

                for chunk in r.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:

                        f.write(chunk)

                        downloaded += len(chunk)

                        pbar.update(
                            len(chunk)
                        )

    print(
        f"✅ Finished {filename} "
        f"({downloaded / (1024**2):.1f} MB)"
    )

print("\n✅ Downloads complete")


⬇️ Downloading Society_3of3.zip


Society_3of3.zip:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

In [28]:
import cv2
import mediapipe as mp
import numpy as np
import os
from tqdm.auto import tqdm

# ============================================
# SETTINGS
# ============================================

DATASET_DIR = "/content/include_selected"

SEQUENCE_LENGTH = 50

# 8 healthcare-related ISL classes
CLASS_NAMES = [
    "Medicine",
    "Hospital",
    "Today",
    "Tomorrow",
    "Doctor",
    "Patient",
    "sick",
    "healthy"
]

LABEL_MAP = {
    name: idx
    for idx, name in enumerate(CLASS_NAMES)
}

print("Classes:")
for name, idx in LABEL_MAP.items():
    print(idx, "→", name)


# ============================================
# FIND VIDEOS
# ============================================

video_files = []

for root, dirs, files in os.walk(DATASET_DIR):

    for file in files:

        if not file.lower().endswith((".mov", ".mp4")):
            continue

        full_path = os.path.join(root, file)

        # Find the class name in the path
        matched_class = None

        for class_name in CLASS_NAMES:

            if f". {class_name}" in full_path:
                matched_class = class_name
                break

        if matched_class is not None:

            video_files.append(
                (
                    full_path,
                    LABEL_MAP[matched_class],
                    matched_class
                )
            )


print("\n==============================")
print("VIDEOS FOUND")
print("==============================")

print("Total:", len(video_files))

for class_name in CLASS_NAMES:
    count = sum(
        c == class_name
        for _, _, c in video_files
    )
    print(f"{class_name:10s}: {count}")


# ============================================
# MEDIAPIPE
# ============================================

BaseOptions = mp.tasks.BaseOptions
VisionRunningMode = mp.tasks.vision.RunningMode
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions

options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="/content/hand_landmarker.task"
    ),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)


# ============================================
# EXTRACT ONE VIDEO
# ============================================

def extract_left_right_landmarks(video_path):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 25

    frames = []

    while True:

        success, frame = cap.read()

        if not success:
            break

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(
            mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )
        )

    cap.release()

    if len(frames) == 0:
        return None

    # Sample exactly 50 frames
    indices = np.linspace(
        0,
        len(frames) - 1,
        SEQUENCE_LENGTH
    ).astype(int)

    selected_frames = [
        frames[i]
        for i in indices
    ]

    sequence = []

    with HandLandmarker.create_from_options(options) as landmarker:

        for frame_idx, mp_image in enumerate(selected_frames):

            timestamp_ms = frame_idx * 40

            result = landmarker.detect_for_video(
                mp_image,
                timestamp_ms
            )

            # --------------------------------
            # LEFT + RIGHT
            # --------------------------------

            frame_landmarks = np.zeros(
                (2, 21, 3),
                dtype=np.float32
            )

            if result.hand_landmarks:

                for hand_idx, hand in enumerate(
                    result.hand_landmarks
                ):

                    if hand_idx >= len(
                        result.handedness
                    ):
                        continue

                    handedness = (
                        result.handedness[hand_idx][0]
                    )

                    hand_label = handedness.category_name

                    # MediaPipe may report:
                    # Left / Right

                    if hand_label == "Left":
                        target_idx = 0

                    elif hand_label == "Right":
                        target_idx = 1

                    else:
                        continue

                    for landmark_idx, landmark in enumerate(hand):

                        frame_landmarks[
                            target_idx,
                            landmark_idx
                        ] = [
                            landmark.x,
                            landmark.y,
                            landmark.z
                        ]

            sequence.append(frame_landmarks)

    return np.array(sequence)


# ============================================
# PROCESS DATASET
# ============================================

X_lr = []
y_lr = []
paths_lr = []

print("\n==============================")
print("EXTRACTING LANDMARKS")
print("==============================")

for video_path, label, class_name in tqdm(
    video_files,
    desc="Processing videos"
):

    try:

        landmarks = extract_left_right_landmarks(
            video_path
        )

        if landmarks is not None:

            X_lr.append(landmarks)
            y_lr.append(label)
            paths_lr.append(video_path)

    except Exception as e:

        print("\n❌ ERROR")
        print(video_path)
        print(e)


# ============================================
# NUMPY ARRAYS
# ============================================

X_lr = np.array(
    X_lr,
    dtype=np.float32
)

y_lr = np.array(
    y_lr,
    dtype=np.int64
)


print("\n==============================")
print("NEW DATASET READY")
print("==============================")

print("X shape:", X_lr.shape)
print("y shape:", y_lr.shape)

print("\nClass distribution:")

for class_name, label in LABEL_MAP.items():

    print(
        f"{class_name:10s}: "
        f"{np.sum(y_lr == label)}"
    )

Classes:
0 → Medicine
1 → Hospital
2 → Today
3 → Tomorrow
4 → Doctor
5 → Patient
6 → sick
7 → healthy

VIDEOS FOUND
Total: 28
Medicine  : 0
Hospital  : 0
Today     : 0
Tomorrow  : 0
Doctor    : 14
Patient   : 14
sick      : 0
healthy   : 0

EXTRACTING LANDMARKS


Processing videos:   0%|          | 0/28 [00:00<?, ?it/s]

KeyboardInterrupt: 